# Global Clinical Trials Registry

Starter notebook for the Kaggle dataset in this folder. It loads `clinical_trials.csv` with pandas and walks through quality checks and a first piece of analysis.

**Files:** `clinical_trials.csv`, `data_dictionary.csv`.

## 1. Load the data

In [ ]:
import pandas as pd

df = pd.read_csv("clinical_trials.csv", low_memory=False)
df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce")
print(df.shape)

## 2. Shape, types and missing values

In [ ]:
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")

missing = df.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0].round(1).to_frame("missing_%").head(15)

## 3. Numeric summary

In [ ]:
df.describe(include='number').T.round(2).head(20)

## 4. Status and phase mix

In [ ]:
df["overall_status"].value_counts().head(10).to_frame("trials")

In [ ]:
(df.groupby(["phase_clean", "overall_status"]).size()
   .unstack(fill_value=0).head(10))

## 5. Who runs trials\n\nLead sponsor class, cleaned up.

In [ ]:
df["lead_sponsor_class"].value_counts().to_frame("trials")

## 6. Reporting lag\n\nDays between completion and first results posting.

In [ ]:
done = df.dropna(subset=["completion_date", "results_first_posted"]).copy()
done["lag_days"] = (pd.to_datetime(done["results_first_posted"], format="mixed", errors="coerce")
                    - pd.to_datetime(done["completion_date"], format="mixed", errors="coerce")).dt.days
done["lag_days"].clip(lower=0).describe().round(0).to_frame("days")

## 7. Enrollment by phase

In [ ]:
df.groupby("phase_clean")["enrollment_count"].median().sort_values().to_frame("median_enrollment")

## 8. Most-studied conditions

In [ ]:
df["conditions"].str.split(";").explode().str.strip().value_counts().head(15).to_frame("trials")

## 9. Trials starting per year

In [ ]:
df["start_year"].dropna().astype(int).value_counts().sort_index().plot(
    kind="line", figsize=(10, 4), title="Trial starts by year (sample)")

## Next steps

See `KAGGLE.md` for analysis ideas, caveats and provenance.